In [2]:
# === Импорты ===
import docx
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LlamaTokenizer

In [3]:
from sentence_transformers import SentenceTransformer, util
from sentence_transformers import CrossEncoder
# Лёгкая многоязычная модель
bi_encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [5]:
from sentence_transformers import CrossEncoder

# NLI cross-encoder
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-base")

In [7]:
import os
import glob
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, util, CrossEncoder
import docx
from striprtf.striprtf import rtf_to_text
!pip install python-docx PyPDF2 striprtf
import PyPDF2
import math

Defaulting to user installation because normal site-packages is not writeable
  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [8]:
def read_docx_tables(path):
    """Извлекает текст из параграфов и таблиц .docx"""
    doc = docx.Document(path)
    parts = []

    # Параграфы
    for p in doc.paragraphs:
        if p.text.strip():
            parts.append(p.text.strip())

    # Таблицы
    for table in doc.tables:
        for row in table.rows:
            row_text = []
            for cell in row.cells:
                if cell.text.strip():
                    row_text.append(cell.text.strip())
            if row_text:
                parts.append(" | ".join(row_text))

    return "\n".join(parts)

In [9]:
class AnswerEvaluator:
    def __init__(self, bi_encoder, cross_encoder, nli_model, jd_file=None, weights=None):
        self.bi_encoder = bi_encoder
        self.cross_encoder = cross_encoder
        self.nli_model = nli_model
        self.weights = weights or {"bi":0.25, "cross":0.25, "nli":0.3, "rag":0.2}
        # JD (один файл)
        self.jd_chunks = []
        self.jd_embeddings = None
        self.jd_source = None
        if jd_file:
            self._load_job_description(jd_file)

    def _normalize_similarity(self, sim):
        return (sim + 1) / 2

    def _keyword_check(self, question, answer):
        q_words = set(w.lower() for w in question.split())
        a_words = set(w.lower() for w in answer.split())
        overlap = q_words & a_words
        return len(overlap) > 0, overlap

    def _load_job_description(self, jd_file):
        """Загружаем один JD-файл и разбиваем на chunks"""
        text = read_docx_tables(jd_file)
        self.jd_source = os.path.basename(jd_file)

        self.jd_chunks = []
        for chunk in text.split("\n"):
            chunk = chunk.strip()
            if len(chunk) > 20:
                self.jd_chunks.append(chunk)

        if self.jd_chunks:
            self.jd_embeddings = self.bi_encoder.encode(
                self.jd_chunks, convert_to_tensor=True
            )

    def _rag_relevance(self, answer, top_k=3):
        if self.jd_embeddings is None:
            return 0.0, []

        emb_a = self.bi_encoder.encode(answer, convert_to_tensor=True)
        hits = util.semantic_search(emb_a, self.jd_embeddings, top_k=top_k)[0]

        # фильтруем по порогу 0.3
        matches = []
        scores = []
        for h in hits:
            if h["score"] >= 0.3:
                matches.append({
                    "source": self.jd_source,
                    "text": self.jd_chunks[h["corpus_id"]],
                    "score": round(float(h["score"]), 3)
                })
                scores.append(h["score"])

        avg_score = float(sum(scores) / len(scores)) if scores else 0.0
        return avg_score, matches

    def evaluate(self, question: str, answer: str):
        # 1. Bi-encoder similarity
        emb_q = self.bi_encoder.encode(question, convert_to_tensor=True)
        emb_a = self.bi_encoder.encode(answer, convert_to_tensor=True)
        similarity = util.cos_sim(emb_q, emb_a).item()
        similarity_norm = self._normalize_similarity(similarity)

        # 2. Cross-encoder similarity
        cross_score = float(self.cross_encoder.predict([(question, answer)])[0])
        cross_score_norm = 1 / (1 + math.exp(-cross_score))

        # 3. NLI entailment prob
        logits = self.nli_model.predict([(question, answer)])[0]
        probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
        entail_prob = float(probs[2])

        # 4. Keyword coverage
        has_keywords, overlap = self._keyword_check(question, answer)
        has_keywords_bonus = 0.05 if has_keywords else 0.0

        # 5. RAG relevance (JD check)
        rag_score, matched_chunks = self._rag_relevance(answer, top_k=3)

        # --- Комбинированный скор ---
        final_score = (
            self.weights['bi'] * similarity_norm +
            self.weights['cross'] * cross_score_norm +
            self.weights['nli'] * entail_prob +
            self.weights['rag'] * rag_score +
            has_keywords_bonus
        )

        # --- Интерпретация ---
        if not has_keywords and entail_prob < 0.3 and rag_score < 0.3:
            verdict = "❌ Не по теме"
            explanation = "Ответ не содержит ключевых слов и не связан с вопросом или JD."
        elif final_score < 0.55:
            verdict = "⚠️ Частичный/общий ответ"
            explanation = "Есть пересечение по смыслу, но ответ недостаточно релевантен или слабая связь с JD."
        else:
            verdict = "✅ Релевантный ответ"
            explanation = "Ответ хорошо согласуется с вопросом и JD."

        return {
            "similarity": round(similarity_norm, 3),
            "cross_score": round(cross_score_norm, 3),
            "entail_prob": round(entail_prob, 3),
            "rag_score": round(rag_score, 3),
            "jd_matches": matched_chunks,
            "keywords": list(overlap),
            "final_score": round(final_score, 3),
            "verdict": verdict,
            "explanation": explanation
        }

In [26]:
evaluator = AnswerEvaluator(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    nli_model=nli_model,
    jd_file = "./kaggle/input/descriptions/Description of Specialist IT.docx"
)

In [27]:
print("Загружено кусков JD:", len(evaluator.jd_chunks))

q = "Как вы решали технические проблемы с серверным оборудованием?"
a = "Мне нравится играть в футбол и читать книги."

print(evaluator.evaluate(q, a))

Загружено кусков JD: 20
{'similarity': 0.45, 'cross_score': 0.996, 'entail_prob': 0.002, 'rag_score': 0.0, 'jd_matches': [], 'keywords': [], 'final_score': 0.362, 'verdict': '❌ Не по теме', 'explanation': 'Ответ не содержит ключевых слов и не связан с вопросом или JD.'}


In [34]:
model_path="./kagglehub/models/nikolayposrednikov/my_model/pytorch/default/7"

tokenizer = AutoTokenizer.from_pretrained(
    model_path
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
# model_name = "mistralai/Mistral-7B"

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )


# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto"
# )


RuntimeError: No GPU found. A GPU is needed for quantization.

In [ ]:
class ChatEngine:
    def __init__(self, model, tokenizer, job_description, candidate_vector, evaluator, device="cuda"):
        self.model = model
        self.tokenizer = tokenizer
        self.job_description = job_description
        self.candidate_vector = candidate_vector
        self.device = device

        self.chat_history = []   # [(роль, текст)]
        self.answer_log = []  # [{question, answer, score, verdict, details}]
        self.summary = ""        # краткое резюме
        self.turns_since_summary = 0
        self.for_promt = {}
        self.evaluator = evaluator

        self.not_answer = False
        self.not_answer_shtraf = 0

    def build_prompt(self, give_promt=None, candidate_emotion: dict = None):
        """Формируем prompt с учётом вакансии, истории и эмоций"""
        history_text = ""
        for role, text in self.chat_history[-6:]:
            history_text += f"{role.upper()}: {text}\n"

        if give_promt is None:
            prompt = f"""
                Ты HR-ассистент, проводишь собеседование.

                Описание вакансии:
                {self.job_description}

                Известные характеристики кандидата:
                {self.candidate_vector}

                Краткое содержание предыдущей беседы:
                {self.summary}

                История последних сообщений:
                {history_text}
            """

            if len(self.for_promt) != 0:
                prompt += f"""
                Ответил на предыдущий вопрос:
                {self.for_promt}
                """

            if candidate_emotion:
                prompt += f"""
                Эмоциональное состояние кандидата (по данным анализа речи):
                {candidate_emotion}
                """

            prompt += f"""
                Задача: сгенерируй только следующий уместный вопрос кандидату на русском языке,
                ориентируясь на требования вакансии, его опыт и настроение.
                Не пиши ответ за кандидата.
                Выведи только вопрос HR.
            """
        else:
            prompt = give_promt

        return prompt


    def clean_hr_question(self, text: str) -> str:
        # Убираем лишние строки и берём только вопрос
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        # ищем все строки с вопросительным знаком
        questions = [l for l in lines if "?" in l]
        if questions:
            return questions[-1]  # берём последний вопрос
        return lines[-1] if lines else text.strip()

    def ask_model(self, prompt, max_new_tokens=150):
        """Генерация из модели"""
        messages = [
            {"role": "system", "content": "Ты — HR для проведения собеседований."},
            {"role": "user", "content": prompt}
        ]
        inputs = self.tokenizer.apply_chat_template(messages, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(
                inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9
            )
        decoded = self.tokenizer.decode(output[0], skip_special_tokens=True)

        if "HR:" in decoded:
            decoded = decoded.split("HR:")[-1].strip()

        #print(f"\n\n{decoded}\n\n")
        return self.clean_hr_question(decoded)

    def summarize_history(self):
        """Сжимает историю диалога в краткое резюме"""
        if len(self.chat_history) < 6:
            return

        history_text = ""
        for role, text in self.chat_history:
            history_text += f"{role}: {text}\n"

        prompt = f"""
            Ты — HR ассистент.
            Сожми кратко диалог ниже, выделив ключевые навыки, опыт и эмоциональные реакции кандидата.
            Не повторяй дословно, делай конспект.

            Диалог:
            {history_text}

            Краткое резюме:
        """

        summary_text = self.ask_model(prompt, max_new_tokens=120)
        self.summary = summary_text.strip()
        self.chat_history = self.chat_history[-4:]  # оставляем только последние ходы
        self.turns_since_summary = 0

    def chat(self, candidate_reply: str = None, candidate_emotion: dict = None):
        """Добавляем ответ кандидата (если есть), генерируем новый вопрос HR"""
        if candidate_reply:
            self.chat_history.append(("CANDIDATE", candidate_reply))
            self.turns_since_summary += 1

            # --- оценка ответа ---
            if len(self.chat_history) >= 2:  # есть вопрос перед ответом
                last_question = self.chat_history[-2][1]
                if last_question and candidate_reply:  # защита от None
                    eval_result = self.evaluator.evaluate(last_question, candidate_reply)
                    self.for_promt = {"Вердикт": eval_result["verdict"], 'Характеристика ответа': eval_result["explanation"]}
                    self.answer_log.append({
                        "question": last_question,
                        "answer": candidate_reply,
                        **eval_result
                    })

                    if eval_result["verdict"].startswith("❌") and not self.not_answer:
                        self.not_answer = True
                        reask_prompt = f"""
                        Ты HR-ассистент. Кандидат ответил не по теме.
                        Тебе нужно переспросить вопрос так, чтобы он стал КОРОЧЕ и ПРОЩЕ.

                        ⚠️ Жёсткие правила:
                        - Всегда начинай переспрос с фразы: "Может вы можете рассказать о..."
                        - Сохраняй только ОДНУ тему из исходного вопроса (не расширяй, не добавляй детали).
                        - Формулируй вопрос максимально просто (1 короткое предложение).
                        - Никаких уточнений, технических параметров, дополнительных условий.
                        - Вопрос должен быть короче исходного.

                        Вот предыдущий вопрос: "{last_question}"

                        Формат вывода:
                        - выведи ТОЛЬКО новый уточняющий вопрос HR,
                        - без пояснений и без кавычек,
                        - вопрос должен заканчиваться знаком "?".
                        """
                        question = self.ask_model(reask_prompt)
                        self.chat_history.append(("HR", question))
                        self.not_answer_shtraf += 0.05
                        return question

        if self.not_answer:
            self.not_answer = False


        # если накопилось много реплик — сжать
        if self.turns_since_summary >= 5:
            self.summarize_history()

        prompt = self.build_prompt(candidate_emotion=candidate_emotion)
        question = self.ask_model(prompt)

        self.chat_history.append(("HR", question))
        return question

    def final_decision(self, threshold=0.6, min_relevant=0.5):
        """Возвращает решение о приёме кандидата"""
        if not self.answer_log:
            return {"decision": "Недостаточно данных", "average_score": 0.0}

        scores = [x["final_score"] for x in self.answer_log]
        sum_scores = sum(scores) - self.not_answer_shtraf
        print(self.not_answer_shtraf)
        avg_score = sum_scores / len(scores)

        relevant_ratio = sum(1 for x in self.answer_log if x["verdict"] == "✅ Релевантный ответ") / len(scores)

        if avg_score >= threshold and relevant_ratio >= min_relevant:
            decision = "✅ Кандидат успешно прошёл собеседование"
        else:
            decision = "❌ Кандидат не прошёл собеседование"

        return {
            "decision": decision,
            "average_score": round(avg_score, 3),
            "relevant_ratio": round(relevant_ratio, 2),
            "answers_evaluated": len(scores)
        }

In [ ]:
import re

with open("/kaggle/input/candidats-summary/Resume 2 Specialist IT_summary.txt", "r", encoding="utf-8") as file:
    candidate_vector = file.readlines()
candidate_vector = "".join([i for i in candidate_vector[1:]])
candidate_vector = candidate_vector.replace("Score", "Оценка cоответствия резюме кандидата вакансии")
persent = int(float((re.search(r"\d+.\d+", candidate_vector)).group()) * 100)
candidate_vector = re.sub(r"\d+.\d+", f"{persent}%", candidate_vector)
print(
    candidate_vector
    )

# === загрузка вакансии ===
job_doc_path = "/kaggle/input/for-llama/Description of Specialist IT.docx"
doc = docx.Document(job_doc_path)
job_description = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

In [ ]:
engine = ChatEngine(model, tokenizer, job_description, candidate_vector, evaluator)